In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import layers

In [3]:
model = load_model("/content/drive/MyDrive/b/modelo_clasificador_transacciones.keras")

with open("/content/drive/MyDrive/b/artefactos_clasificacion.pkl", "rb") as f:
    artefactos = pickle.load(f)

label_encoder_macro = artefactos['label_encoder_macro']
label_encoder_micro = artefactos['label_encoder_micro']
vocabulario = artefactos['config_vectorizador']['vocabulario']

print(f"Clases macro: {list(label_encoder_macro.classes_)}")
print(f"Clases micro: {list(label_encoder_micro.classes_)}")

Modelo y artefactos cargados correctamente.
Clases macro: ['Alimentacion', 'Entretenimiento', 'Finanzas', 'Hogar', 'Salud', 'Transporte']
Clases micro: ['alquiler_y_expensas', 'atencion_medica', 'carniceria_y_granja', 'cobertura_medica', 'combustible', 'cuidado_personal', 'delivery', 'farmacia', 'hobbies_y_deportes', 'impuestos', 'indumentaria', 'mantenimiento_vehicular', 'mantenimiento_y_muebles', 'pago_tarjetas', 'peajes', 'restaurante', 'servicios_basicos', 'supermercado', 'suscripciones_digitales', 'taxi_y_apps', 'transferencias', 'transporte_publico']


In [4]:
max_tokens = 5000
sequence_length = 5

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.set_vocabulary(vocabulario)

print(f"Vocabulario cargado: {len(vocabulario)} tokens")

Vocabulario cargado: 136 tokens


In [8]:
def predecir_transaccion(nombre_tienda, contexto=""):
    """
    Recibe el nombre de la tienda/comercio y opcionalmente un contexto,
    y devuelve la categoría macro y micro predichas con su confianza.
    """
    input_nombre = tf.constant([nombre_tienda], dtype=tf.string)
    input_contexto = tf.constant([contexto if contexto else nombre_tienda], dtype=tf.string)

    pred_macro, pred_micro = model.predict(
        {'input_nombre': input_nombre, 'input_contexto': input_contexto},
        verbose=0
    )

    idx_macro = np.argmax(pred_macro[0])
    idx_micro = np.argmax(pred_micro[0])

    categoria_macro = label_encoder_macro.inverse_transform([idx_macro])[0]
    categoria_micro = label_encoder_micro.inverse_transform([idx_micro])[0]

    confianza_macro = pred_macro[0][idx_macro]
    confianza_micro = pred_micro[0][idx_micro]

    return {
        'categoria_macro': categoria_macro,
        'confianza_macro': float(confianza_macro),
        'categoria_micro': categoria_micro,
        'confianza_micro': float(confianza_micro)
    }

In [15]:
resultado = predecir_transaccion("Netflix", contexto="suscripciones_digitales")

print(f"Categoría macro: {resultado['categoria_macro']} ({resultado['confianza_macro']:.2%})")
print(f"Categoría micro: {resultado['categoria_micro']} ({resultado['confianza_micro']:.2%})")

Categoría macro: Entretenimiento (100.00%)
Categoría micro: suscripciones_digitales (99.98%)
